# Introducción a la Programación Orientada a Objetos (POO)

La POO se basa en la creación de clases y objetos para organizar el código de manera modular. En este proyecto, la POO se utiliza para organizar las distintas funcionalidades del bot en clases independientes, lo que facilita el mantenimiento y la expansión del código.

## Conceptos clave:
- **Clases**: Plantillas que definen los comportamientos y atributos de los objetos.
- **Objetos**: Instancias de clases que contienen datos y comportamientos específicos.
- **Encapsulamiento**: Ocultar los detalles de implementación y exponer solo lo necesario.
- **Abstracción**: Definir interfaces claras para interactuar con el código sin necesidad de conocer su implementación interna.


# Descripción General del Proyecto

Este proyecto es un bot de Telegram que utiliza un modelo de Machine Learning para clasificar preguntas y dar respuestas. El código está estructurado de manera orientada a objetos, utilizando clases como `Preprocesador` y `ClasificadorBot` para modularizar las diferentes funcionalidades.

En esta sección, exploraremos cómo el código está organizado utilizando POO para gestionar las interacciones del bot.


# Librerias y configuracion


```python
import os
import re
import joblib
import unicodedata
from dotenv import load_dotenv
from telegram import Update
from telegram.ext import ApplicationBuilder, MessageHandler, ContextTypes, filters
from logger import setup_logger

**OS**

Proporciona una forma de interactuar con el sistema operativo desde Python. Permite trabajar con rutas de archivos, directorios, y también proporciona acceso a funcionalidades relacionadas con el entorno de ejecución, como la gestión de variables de entorno y la ejecución de comandos del sistema.

Usos comunes:

- Acceder a rutas de archivos y directorios (os.path).

- Leer y escribir variables de entorno (os.getenv).

- Manejar procesos del sistema (os.system).


**RE**

Librería de expresiones regulares en Python. Permite trabajar con patrones de texto para buscar, reemplazar o manipular cadenas de caracteres de manera eficiente y flexible.

Usos comunes:

- Buscar patrones en texto (re.search, re.match).

- Reemplazar partes de texto utilizando patrones (re.sub).

- Dividir texto según expresiones regulares (re.split).

- Validar formatos de datos como emails, números de teléfono, etc


**JOBLIB**

Librería especializada en la serialización (guardar y cargar) de objetos en Python, especialmente útil para modelos de Machine Learning. Permite guardar objetos complejos, como modelos entrenados, de manera eficiente en disco y luego cargarlos sin perder su funcionalidad.

Usos comunes:

- Guardar y cargar modelos de Machine Learning entrenados (por ejemplo, con joblib.dump y joblib.load).

- Serializar otros objetos como diccionarios grandes o matrices.

**UNICODEDATA**

Proporciona acceso a la base de datos Unicode y permite trabajar con caracteres Unicode. Ofrece herramientas para normalizar y analizar caracteres, como separar caracteres combinados (acentos) y verificar categorías de caracteres (letras, dígitos, etc.).

Usos comunes:

- Normalización de caracteres (unicodedata.normalize).

- Comprobar categorías de caracteres (por ejemplo, si es una letra o un signo de puntuación) con unicodedata.category.

- Obtener información detallada sobre caracteres Unicode.

**FROM DOTENV IMPORT LOAD_DOTENV**

Librería que facilita el manejo de variables de entorno en un archivo .env. load_dotenv se utiliza para cargar las variables de entorno desde un archivo .env en el entorno de ejecución de Python, lo que permite almacenar datos sensibles (como claves de API o contraseñas) sin hardcodearlos en el código.

Usos comunes:

- Cargar variables de entorno desde un archivo .env con load_dotenv().

- Utilizar variables de entorno en el código con os.getenv("NOMBRE_DE_LA_VARIABLE").

**FROM TELEGRAM IMPORT UPDATE**

Es una clase de la librería python-telegram-bot que contiene la información sobre un mensaje recibido por el bot de Telegram. Esto incluye detalles como el texto del mensaje, el remitente, el chat, entre otros.

Usos comunes:

- Acceder a los detalles de un mensaje recibido (update.message.text, update.message.chat_id).

- Extraer información del usuario que interactúa con el bot (como el ID del usuario).

**FROM TELEGRAM.EXT IMPORT APLICATTIONBUILDER, MESSAGEHANDLER, CONTEXTYPES, FILTERS**

Estas clases y funciones son parte de la librería python-telegram-bot y están diseñadas para facilitar la interacción con la API de Telegram, especialmente en lo que respecta a manejar mensajes y eventos.

- ApplicationBuilder: Se usa para construir la aplicación de Telegram, configurando el token y otros parámetros necesarios.

- MessageHandler: Se usa para manejar mensajes específicos que el bot recibe. Permite definir cómo debe responder el bot a mensajes que coinciden con ciertos filtros.

- ContextTypes: Define los tipos de contexto que se pasan entre los diferentes manejadores, como DEFAULT_TYPE, que es el tipo de contexto por defecto que contiene información de la interacción.

- filters: Permite definir filtros para manejar diferentes tipos de mensajes, como mensajes de texto, comandos, etc. Un filtro común sería filters.TEXT, que solo permite que el bot responda a mensajes de texto.



**FROM LOGGER IMPORT SETUP_LOGGER**

Esta línea importa una función llamada setup_logger de un módulo llamado logger. setup_logger se encarga de configurar el registro de eventos (logging), permitiendo a la aplicación registrar mensajes de depuración, advertencias y errores.

Usos comunes:

- Configurar un sistema de registro de logs para la aplicación.

- Definir cómo y dónde almacenar los logs (por ejemplo, en un archivo de texto o en una base de datos).

- Usar los logs para depurar, monitorear y obtener información sobre el comportamiento del sistema

# Clase Preprocesador 

La clase `Preprocesador` se encarga de limpiar y preparar el texto que el bot recibirá para devolverselo limpio. A continuación, desglosamos su método `limpiar_texto`.

**Método `limpiar_texto`**
Este método realiza varias transformaciones en el texto de entrada para hacerlo más adecuado para el procesamiento.

Este método toma un texto como entrada y realiza las siguientes transformaciones:
1. Convierte todo el texto a minúsculas.
2. Elimina los acentos y caracteres diacríticos (tildes).
3. Elimina todos los caracteres que no sean alfanuméricos o espacios.
4. Reemplaza los espacios redundantes por un solo espacio.

In [ ]:
class Preprocesador:
    def limpiar_texto(self, texto: str) -> str:
        texto = texto.lower() #De un "Hola Mundo" devuelve un "hola mundo"
        texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn') #Si se escribe "año" devuelve "ano"
        texto = re.sub(r'[^\w\s]', '', texto) #De un "¡Hola, mundo!" devuelve un "Hola Mundo"
        texto = re.sub(r'\s+', ' ', texto).strip() #De un "    Hola   Mundo " devuelve un "Hola Mundo"
        return texto

'''
1- Convertir a minúsculas (texto = texto.lower()):
Convierte todo el texto a minúsculas para evitar que las diferencias de mayúsculas y minúsculas afecten al análisis posterior.

2- Eliminar acentos (''.join(...)):
Utiliza unicodedata.normalize para descomponer los caracteres acentuados en su forma base y el signo diacrítico (la tilde). Luego, filtra solo los caracteres alfabéticos y numéricos, eliminando así las tildes y otros caracteres diacríticos.

3- Eliminar caracteres no alfanuméricos (re.sub(...)):
Usa una expresión regular para eliminar cualquier carácter que no sea alfanumérico o espacio. Esto ayuda a limpiar símbolos innecesarios como signos de puntuación.

4- Reemplazar múltiples espacios (re.sub(r'\s+', ' ', texto)):
Reemplaza cualquier secuencia de espacios consecutivos por un solo espacio. Luego, strip() elimina los espacios al inicio y al final.
'''

# Clase `ClasificadorBot`

La clase `ClasificadorBot` es la encargada de manejar la interacción con el modelo de Machine Learning y la API de Telegram. Esta clase tiene dos responsabilidades principales:
1. Cargar el modelo entrenado y el vectorizador.
2. Preprocesar las preguntas de los usuarios y generar respuestas utilizando el modelo.
3. Para esto toma la pregunta del usuario, le aplica los metodos de limpieza del Preprocesador y convierte el texto procesado en una representacion numerica

**Constructor `__init__`**

El constructor `__init__` inicializa el bot, cargando el modelo y el vectorizador desde archivos, y también configura el preprocesador y el logger.

In [ ]:
class ClasificadorBot:
    def __init__(self, modelo_path, vectorizer_path): #El metodo constructor recibe las rutas donde se encuentra el modelo entrenado y el vectorizador
        self.modelo = joblib.load(modelo_path)  # Cargar el modelo entrenado
        self.vectorizer = joblib.load(vectorizer_path)  # Cargar el vectorizador
        self.preprocesador = Preprocesador()  # Instanciar el preprocesador
        self.logger = setup_logger()  # Configurar el logger

'''
1- Cargar el modelo y vectorizador (joblib.load):
Estos archivos contienen el modelo entrenado y el vectorizador previamente entrenados y guardados. El modelo se usa para realizar las predicciones sobre el texto, y el vectorizador convierte el texto en una representación numérica.

2- Instanciar el preprocesador:
El preprocesador se encargará de limpiar las preguntas de los usuarios antes de que sean pasadas al modelo.

3- Configurar el logger:
El logger se configura para registrar eventos y errores del sistema.
'''

# Método `obtener_respuesta`

El método `obtener_respuesta` toma una pregunta del usuario, la procesa y genera una respuesta utilizando el modelo de Machine Learning.

In [ ]:
def obtener_respuesta(self, pregunta_usuario: str) -> str: #El metodo obtiene una entrada de str y devuelve un str
        pregunta_limpia = self.preprocesador.limpiar_texto(pregunta_usuario)  # Limpiar el texto, pasando la pregunta del usuario al metodo limpiar_texto
        pregunta_vectorizada = self.vectorizer.transform([pregunta_limpia])  # Vectorizar el texto transformandolo en una representacion numerica que pueda entender el modelo
        respuesta = self.modelo.predict(pregunta_vectorizada)  # Obtener la predicción btemiendo una pregunta vectorizada y prediciendo cual es la respuesta o la clase asociada
        return respuesta[0]  # Devolver la respuesta (0, la primera) de la prediccion obtenida. Esta es la respuesta al usuario

'''
1- Limpiar el texto (pregunta_limpia):
El texto de la pregunta del usuario se pasa al preprocesador para limpiarlo antes de cualquier análisis.

2- Vectorizar el texto (pregunta_vectorizada):
El texto limpio se convierte en una representación numérica utilizando el vectorizador, lo que es necesario para que el modelo pueda hacer una predicción.

3- Obtener la respuesta del modelo:
El modelo predice la categoría o respuesta asociada al texto vectorizado. El resultado es un array, y por eso se devuelve el primer elemento (respuesta[0]).
'''

# Método `responder`

El método `responder` es responsable de manejar los mensajes recibidos por el bot de Telegram. Procesa el mensaje, valida si tiene sentido y genera una respuesta utilizando el modelo.

In [ ]:
async def responder(self, update: Update, context: ContextTypes.DEFAULT_TYPE): # METODO ASINCRONICO PARA INTERACTUAR CON EL USUARIO, RECIBIR LOS MENSAJES, PROCESARLOS Y DEVOLVER UNA RESPUESTA
    try: #El codigo se envuelve en un try para manejar los posibles errores sin afectar la ejecucion del metodo, ya que si esto ocurre, dicho error se captura sin interrumpir el flujo
        pregunta = update.message.text #Obtencion del mensaje
        mensaje_limpio = self.preprocesador.limpiar_texto(pregunta)#Limpieza del mensaje
        #Defino variables a analizar:
        es_muy_corto = len(mensaje_limpio) < 2 #mide la longitud del mensaje (no puede ser mas corto que 2)
        solo_una_letra_o_numero = mensaje_limpio.isalum() and len(mensaje_limpio == 1) #verifica que el mensaje no sea solo un caracter alfanumerico (a o 5)
        contiene_palabras = any(c.isalpha() for c in mensaje_limpio) and len(mensaje_limpio.split()) >= 1 #corrobora que al menos uno de los caracteres sea una letra y que el texto tenga al menos una palabra
        
        if es_muy_corto or solo_una_letra_o_numero or not contiene_palabras:
            await update.message.reply_text("Por ahí quisiste decir otra cosa o no entendí🧐, ¿lo escribís de nuevo?")
            return
        
        respuesta = self.obtener_respuesta(pregunta) #si el mensaje es valido se obtiene una respuesta
        await update.message.reply_text(f"{respuesta}") # una vez obtenida la respuesta el bot la envia al usuario con await para que el mensaje se envie de manera asincronica para evitar bloquear otras interacciones
        
    except Exception as e: #si ocurre algun error en el codigo se captura con el except y se guarda en el logger
        self.logger.error(f"(Error al responder: {e})")
        await update.message.reply_text("Si te digo que se cayó el sistema, ¿me creés? jaja😂.")#respuesta al usuario cuando hay un error)
        

'''
1- Obtener y limpiar el mensaje:
El mensaje del usuario se obtiene de update.message.text, luego se limpia utilizando el preprocesador 

2- Validaciones del mensaje:
Se verifica que el mensaje no sea demasiado corto, que no sea solo una letra o número, y que contenga palabras válidas. Si no cumple con estas condiciones, el bot pide al usuario que reformule su mensaje.

3- Obtener y enviar la respuesta:
Si el mensaje es válido, se obtiene la respuesta del modelo y se envía al usuario.
'''

# Método `main`

El método `main` es el punto de entrada para iniciar el bot. Configura y arranca la aplicación de Telegram, y define cómo se manejarán los mensajes.

In [ ]:
async def main():#metodo principal que se ejecuta cuando se inicia el bot. esta en asyncro para no frenar y bloquear ejecuciones mientras se espera la pregunta 
    load_dotenv()#carga las variables (token) que estan fuera del codigo
    telegram_bot_token = os.getenv("telegram_bot_token")
    
    if not telegram_bot_token:#si no se encuentra el token devuelve un mensaje de error una excepcion(valuerror)
        raise ValueError("No se encontro el token en el archivo .env")
    
    modelo_path = "../modelo/modelo_entrenado.pkl"
    vectorizer_path = "../modelo/vectorizer.pkl"
    
    bot = Clasificador_Bot("../modelo/modelo_entrenado.pkl", "../modelo/vectorizer.pkl")#se crea una instancia del bot (clasificador_bot) con las rutas de los archivos del moedelo y el vectorizador
    
    app = ApplicationBuilder().token(telegram_bot_token).build()# se construye la aplicacion del bot utilizando la libreria de python telegram bot y pasandole el token
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, bot.responder))#se agrega un manejador de mensajes especificando que el bot solo debe responder a texto y no a comandos
    print("🤖 Bot en funcionamiento...") #inicializa el bot
    await app.initialize() #inicializa la aplicacion de telegram de forma asincronica
    await app.start()#inicia la escucha del bot
    await app.updater.start_polling()#hace el polling de la api de telegam significa que el bot estara verificando continuamente si hay mensajes para procesar
    

'''
1- Cargar las variables de entorno:
Se carga el archivo .env para obtener el token del bot, lo que permite que el bot se conecte a la API de Telegram.

2- Configurar y arrancar la aplicación de Telegram:
Se configura la aplicación de Telegram y se configura el manejador de mensajes para que el bot responda a los mensajes de texto.
'''

# El bloque if __name__ == "__main__" 

Asegura que el bot solo se inicie si este archivo se ejecuta directamente (y no cuando se importa desde otro módulo). Se utiliza programación asíncrona para mantener el bot corriendo indefinidamente.

In [ ]:
if __name__ == "__main__":
    import asyncio

    async def safe_main():#se define nuevamente la funcion asyncronica
        await main()#llama a main para inicializar el bot y esperar indefinidamente
        await asyncio.Event().wait()#permite que el bot siga funcionando mientras espera mensajes

    try:
        asyncio.get_event_loop().run_until_complete(safe_main())#permite que el bo no se bloquee con un bucle de eventos
    except KeyboardInterrupt:#el bucle continua hasta que se produce una excepcion por teclado (keyboardinterrump) con ctrl + c
        print("🔴 Bot detenido manualmente.")
        

'''
1- if __name__ == "__main__":
Esta condición garantiza que el bloque se ejecute solo si el script no fue importado como módulo.
Es una buena práctica para modularizar el código y reutilizar funciones sin que se ejecuten automáticamente.

2- import asyncio
Se importa la librería estándar asyncio, que permite ejecutar código asincrónico como tareas en paralelo (sin bloquear el flujo del programa).

3- async def safe_main():
Se define una función asíncrona safe_main() que realiza dos tareas:
Llama a main() para inicializar y arrancar el bot.
Usa await asyncio.Event().wait() para mantener al bot corriendo sin terminar (queda a la espera de eventos, como nuevos mensajes en Telegram).

4- asyncio.get_event_loop().run_until_complete(safe_main())
Ejecuta la función safe_main() dentro del bucle de eventos de asyncio.
Este bucle gestiona la ejecución de todas las tareas asíncronas del bot.

5- except KeyboardInterrupt:
Si el usuario presiona Ctrl + C, se lanza la excepción KeyboardInterrupt.
Se captura la excepción y se muestra un mensaje indicando que el bot fue detenido manualmente.
'''